
# 03 | Data Quality Assessment

**Objective:** Evaluate the completeness, uniqueness, and consistency of the World Cup datasets using SQL.

I perform data quality checks to identify potential issues before conducting more detailed team and player analysis.



## 3.1 | Connecting to the Database

I connect to my existing SQLite database to assess the quality of the imported datasets.


In [1]:

import pandas as pd
import sqlite3
from pathlib import Path

# Connect to existing SQLite database
db_path = Path.cwd().parent / "Data" / "Prepared" / "world_cup_2026.db"

conn = sqlite3.connect(db_path)

print("Connected to World Cup database!")


Connected to World Cup database!



## 3.2 | Checking Missing Values

I check key match fields for missing values to assess whether the dataset is complete enough for reliable analysis.

In [2]:

# Check missing values in the matches table
query = """
SELECT
    COUNT(*) AS total_matches,
    SUM(CASE WHEN match_id IS NULL THEN 1 ELSE 0 END) AS missing_match_id,
    SUM(CASE WHEN home_team IS NULL THEN 1 ELSE 0 END) AS missing_home_team,
    SUM(CASE WHEN away_team IS NULL THEN 1 ELSE 0 END) AS missing_away_team,
    SUM(CASE WHEN home_score IS NULL THEN 1 ELSE 0 END) AS missing_home_score,
    SUM(CASE WHEN away_score IS NULL THEN 1 ELSE 0 END) AS missing_away_score,
    SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS missing_date
FROM matches;
"""

display(pd.read_sql_query(query, conn))


,total_matches,missing_match_id,missing_home_team,missing_away_team,missing_home_score,missing_away_score,missing_date
0,104,0,0,0,0,0,0



## 3.3 | Checking Duplicate Records

I check for duplicate identifiers across the main datasets to verify record uniqueness and prevent inaccurate results during analysis.


In [3]:

# Check duplicate identifiers in key tables
query = """
SELECT
    'matches' AS dataset,
    COUNT(*) AS total_records,
    COUNT(DISTINCT match_id) AS unique_ids,
    COUNT(*) - COUNT(DISTINCT match_id) AS duplicate_or_null_ids
FROM matches

UNION ALL

SELECT
    'teams',
    COUNT(*),
    COUNT(DISTINCT team_id),
    COUNT(*) - COUNT(DISTINCT team_id)
FROM teams

UNION ALL

SELECT
    'players',
    COUNT(*),
    COUNT(DISTINCT player_id),
    COUNT(*) - COUNT(DISTINCT player_id)
FROM players;
"""

display(pd.read_sql_query(query, conn))


,dataset,total_records,unique_ids,duplicate_or_null_ids
0,matches,104,104,0
1,teams,48,48,0
2,players,1277,1277,0



## 3.4 | Validating Match Scores

I validate match scores to identify negative values, missing results, and potentially unusual scoring patterns that may require further investigation.


In [4]:

# Validate match scores
query = """
SELECT
    COUNT(*) AS total_matches,

    SUM(CASE
        WHEN home_score < 0 OR away_score < 0
        THEN 1 ELSE 0
    END) AS negative_scores,

    SUM(CASE
        WHEN home_score IS NULL OR away_score IS NULL
        THEN 1 ELSE 0
    END) AS missing_scores,

    MAX(home_score) AS highest_home_score,
    MAX(away_score) AS highest_away_score,

    MAX(home_score + away_score) AS highest_combined_score

FROM matches;
"""

display(pd.read_sql_query(query, conn))


,total_matches,negative_scores,missing_scores,highest_home_score,highest_away_score,highest_combined_score
0,104,0,0,7,6,10



## 3.5 | Validating Match and Team Consistency

I compare match-level and team-level records to verify that participating teams and their scores are consistent across both datasets.


In [5]:

# Compare match results with team-level records
query = """
SELECT
    COUNT(*) AS total_team_records,

    SUM(CASE
        WHEN m.match_id IS NULL
        THEN 1 ELSE 0
    END) AS unmatched_matches,

    SUM(CASE
        WHEN mt.team_id NOT IN (m.home_team_id, m.away_team_id)
        THEN 1 ELSE 0
    END) AS incorrect_team_ids,

    SUM(CASE
        WHEN mt.home_away = 'home' AND mt.score != m.home_score
        THEN 1
        WHEN mt.home_away = 'away' AND mt.score != m.away_score
        THEN 1
        ELSE 0
    END) AS score_mismatches

FROM match_teams mt
LEFT JOIN matches m
    ON mt.match_id = m.match_id;
"""

display(pd.read_sql_query(query, conn))


,total_team_records,unmatched_matches,incorrect_team_ids,score_mismatches
0,208,0,0,0



## 3.6 | Validating Player Appearances

I verify that player appearance records are linked to valid players, matches, and participating teams before using them for individual performance analysis.


In [6]:

# Validate player appearance relationships
query = """
SELECT
    COUNT(*) AS total_appearances,

    SUM(CASE WHEN p.player_id IS NULL
        THEN 1 ELSE 0 END) AS unmatched_players,

    SUM(CASE WHEN m.match_id IS NULL
        THEN 1 ELSE 0 END) AS unmatched_matches,

    SUM(CASE WHEN mt.match_team_id IS NULL
        THEN 1 ELSE 0 END) AS unmatched_match_teams,

    SUM(CASE
        WHEN p.player_id IS NOT NULL
         AND ma.team_id != p.team_id
        THEN 1 ELSE 0
    END) AS player_team_mismatches

FROM match_appearances ma

LEFT JOIN players p
    ON ma.player_id = p.player_id

LEFT JOIN matches m
    ON ma.match_id = m.match_id

LEFT JOIN match_teams mt
    ON ma.match_team_id = mt.match_team_id;
"""

display(pd.read_sql_query(query, conn))


,total_appearances,unmatched_players,unmatched_matches,unmatched_match_teams,player_team_mismatches
0,5392,0,0,0,0



## 3.7 | Reviewing Data Sources

I review the source references associated with match records to assess data traceability and identify any limitations that should be documented before reporting results.


In [7]:

# Review match source references
query = """
SELECT
    COUNT(*) AS total_matches,
    COUNT(DISTINCT source_url) AS unique_sources,
    SUM(CASE
        WHEN source_url IS NULL OR TRIM(source_url) = ''
        THEN 1 ELSE 0
    END) AS missing_sources
FROM matches;
"""

display(pd.read_sql_query(query, conn))

# Inspect sample source references
query = """
SELECT DISTINCT source_url
FROM matches
WHERE source_url IS NOT NULL
LIMIT 5;
"""

display(pd.read_sql_query(query, conn))


,total_matches,unique_sources,missing_sources
0,104,104,0


,source_url
0,https://www.fifatrainingcentre.com/media/nativ...
1,https://www.fifatrainingcentre.com/media/nativ...
2,https://www.fifatrainingcentre.com/media/nativ...
3,https://www.fifatrainingcentre.com/media/nativ...
4,https://www.fifatrainingcentre.com/media/nativ...



## 3.8 | Data Quality Findings

My validation checks produced the following results:

- **Completeness:** No missing values in the key match fields tested.
- **Uniqueness:** No duplicate or null identifiers in the matches, teams, and players tables.
- **Score validity:** No missing or negative match scores.
- **Referential integrity:** No unmatched records or inconsistencies in the relationships tested.
- **Source traceability:** All 104 matches contain source URLs.

**Limitations:** These checks establish internal consistency but do not independently verify the accuracy of match results or the contents of the source references. Additional validation may be needed before treating the dataset as an authoritative source.
